# 3D Reconstruction with Gaussian Splatting — Step by Step

**Runs on your host machine.** Docker is used only for the two heavy steps:

| Step | Where it runs |
|------|--------------|
| 0 — Build Docker images | host (cell below, run once) |
| 1 — Configure paths | host |
| 2 — Capture / load images | host |
| 3 — COLMAP | `colmap` container |
| 4 — Filter scene (BiRefNet) | host |
| 5 — Train 3DGS-MCMC | `gs` container |
| 6–8 — Filter, scale, mesh | host |

Install host dependencies once: `pip install -r src/science_jubilee/Vision/GS_Reconstruction/requirements.txt`

In [8]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "science_jubilee").is_dir():
        REPO_ROOT = parent
        break
else:
    raise RuntimeError("Could not locate the science_jubilee repository root")

SRC_ROOT = REPO_ROOT / "src"
for path in (SRC_ROOT, REPO_ROOT):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from science_jubilee.Vision.GS_Reconstruction.ingredients.colmap import run_colmap
from science_jubilee.Vision.GS_Reconstruction.ingredients.pre_process import run_filter_scene
from science_jubilee.Vision.GS_Reconstruction.ingredients.reconstruction import run_reconstruction
from science_jubilee.Vision.GS_Reconstruction.ingredients.post_process import run_filter_plants
from science_jubilee.Vision.GS_Reconstruction.ingredients.scaling import run_scale_by_cameras
from science_jubilee.Vision.GS_Reconstruction.ingredients.meshing import run_meshing
from science_jubilee.scripts.ingredients.snake_scan import run_scan


def show(img, title=""):
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()


hardware = False

## 0 - Build Docker images (run once)

Builds the `colmap` and `gs` images. Safe to re-run — Docker caches unchanged layers.

In [10]:
import subprocess
from pathlib import Path

_compose = str(Path(REPO_ROOT) / "src/science_jubilee/Vision/GS_Reconstruction/docker-compose.yml")

proc = subprocess.Popen(
    ["docker", "compose", "-f", _compose, "build"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"docker compose build failed (exit {proc.returncode})")
print("Docker images ready.")

#1 [internal] load local bake definitions
#1 reading from stdin 1.21kB done
#1 DONE 0.0s

#2 [colmap internal] load build definition from Dockerfile.colmap
#2 transferring dockerfile: 328B 0.0s done
#2 DONE 0.1s

#3 [gs internal] load build definition from Dockerfile
#3 transferring dockerfile: 5.69kB 0.0s done
#3 DONE 0.1s

#4 [colmap internal] load metadata for docker.io/colmap/colmap:20250530.2938
#4 ...

#5 [gs internal] load metadata for docker.io/nvidia/cuda:11.7.1-devel-ubuntu20.04
#5 DONE 1.1s

#6 [gs internal] load .dockerignore
#6 transferring context: 399B 0.0s done
#6 DONE 0.0s

#7 [gs internal] load build context
#7 DONE 0.0s

#8 [gs  1/18] FROM docker.io/nvidia/cuda:11.7.1-devel-ubuntu20.04@sha256:47bad1799ade862fa2486b6e5b19ce91c29396f4ca8d74d83b6c228222b6079f
#8 resolve docker.io/nvidia/cuda:11.7.1-devel-ubuntu20.04@sha256:47bad1799ade862fa2486b6e5b19ce91c29396f4ca8d74d83b6c228222b6079f
#8 resolve docker.io/nvidia/cuda:11.7.1-devel-ubuntu20.04@sha256:47bad1799ade862fa24

UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 136: character maps to <undefined>

## 1 - Configure the dataset and scan

The scan captures a serpentine grid of images. Image names contain the capture index and the physical X/Y/Z position. Set `run_capture=True` only when the Jubilee hardware is connected; otherwise the pipeline reuses images already present in `images_dir`.

In [ ]:
dataset_name = "Plante_1" if not hardware else "Latest_reconstruction"
start = [110.0, 80.0, 280.0]
stop = [250.0, 200.0, 220.0]
steps = [5, 5, 4]
delay = 2.0
iterations = 7000
run_capture = hardware

dataset_path = REPO_ROOT / "src/science_jubilee/Vision/GS_Reconstruction/Datasets" / dataset_name
images_dir = dataset_path / "input"
output_path = REPO_ROOT / "src/science_jubilee/Vision/GS_Reconstruction/Outputs" / f"{dataset_name}_results"
output_reconstruction = output_path / "3d_reconstruction"
images_dir.mkdir(parents=True, exist_ok=True)
output_path.mkdir(parents=True, exist_ok=True)

## 2 - Capture the snake scan

`run_scan` moves through the configured X/Y grid, captures one image at each position, and saves coordinate-aware names in the dataset input folder. With `run_capture=False`, this cell verifies that input images already exist instead of moving the machine.

In [5]:
if run_capture:
    saved_images = run_scan(
        start=start,
        stop=stop,
        steps=steps,
        delay=delay,
        out=str(images_dir),
    )
else:
    saved_images = sorted(str(path) for path in images_dir.glob("*.jpg"))
    if not saved_images:
        raise FileNotFoundError(f"No input images found in {images_dir}")

print(f"Using {len(saved_images)} images from {images_dir}")

Using 75 images from C:\Users\Alienor\Documents\Projects\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Datasets\Plante_1\input


## 3 - Prepare COLMAP camera data

COLMAP detects image features and estimates camera poses from the captured input images. The ingredient runs the repository's WSL helper script and creates the dataset's `images` and camera metadata.

In [6]:
run_colmap(dataset_path=dataset_path)

RuntimeError: COLMAP step failed (exit 1)

## 4 - Filter the reconstruction scene

The preprocessing step removes the tray/background from the COLMAP image set. It prepares the scene images used by Gaussian Splatting and can use the repository's AI-based filtering.

In [25]:
run_filter_scene(
    images_path=dataset_path / "images",
    use_ai=True,
)

Loading weights:   0%|          | 0/754 [00:00<?, ?it/s]

True

## 5 - Train the Gaussian Splatting model

This stage optimizes the Gaussian scene representation from the COLMAP cameras and filtered images. The trained point cloud is written under the selected iteration directory.

In [ ]:
run_reconstruction(
    dataset_path=dataset_path,
    output_path=output_reconstruction,
    iterations=iterations,
)

KeyboardInterrupt: 

In [15]:
input_ply = output_reconstruction / "point_cloud" / f"iteration_{iterations}" / "point_cloud.ply"
print(input_ply)

c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\3d_reconstruction\point_cloud\iteration_7000\point_cloud.ply


## 6 - Remove non-plant Gaussians

The post-processing ingredient filters the trained point cloud using geometric, opacity, and color thresholds. The result is saved as the input for camera-based scaling.

In [16]:
filtered_ply = output_reconstruction / "point_cloud" / "iteration_35000" / "point_cloud.ply"
run_filter_plants(
    input_ply=input_ply,
    output_ply=filtered_ply,
        bbox_size=10000,
        bbox_center=[0.0, 2, 0.0],
        elongation_threshold=7.0,
        scale_threshold=1,
        std_ratio=3,
        opacity_threshold=0.07,
        nb_neighbors=60,
        white_sat_thresh=0.55,
        white_val_thresh=0.2,
)
print(filtered_ply)

Loading Gaussians from c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\3d_reconstruction\point_cloud\iteration_7000\point_cloud.ply...
Original splats: 308691
Splats restants après Bounding Box et Propriétés : 18084
Analyse spatiale en cours (SOR)...
Splats finaux conservés : 17855
--> Total supprimé : 290836 splats.
Saved filtered splats to c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\3d_reconstruction\point_cloud\iteration_35000\point_cloud.ply
c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\3d_reconstruction\point_cloud\iteration_35000\point_cloud.ply


## 7 - Scale and align the point cloud

Camera poses and the known scan geometry provide the scale and orientation of the filtered point cloud. This creates a scaled PLY file using the reconstruction camera metadata.

In [17]:
scaled_ply = output_reconstruction / "point_cloud" / "iteration_35000" / "point_cloud_scaled.ply"
run_scale_by_cameras(
    input_ply=filtered_ply,
    output_ply=scaled_ply,
    cameras_json_path=output_reconstruction / "cameras.json",
    cameras_span=None,
)
print(scaled_ply)

True Span found:[0.14, 0.12, 0.06]
Chargement des Gaussiennes depuis c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\3d_reconstruction\point_cloud\iteration_35000\point_cloud.ply...
Fichier final sauvegardé : c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\3d_reconstruction\point_cloud\iteration_35000\point_cloud_scaled.ply
c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\3d_reconstruction\point_cloud\iteration_35000\point_cloud_scaled.ply


## 8 - Build the mesh

The final ingredient converts the scaled Gaussian point cloud into an OBJ mesh using an alpha shape and optional decimation. The mesh is written beside the reconstruction output.

In [18]:
mesh_path = output_path / "mesh.obj"
run_meshing(
    input_ply=scaled_ply,
    output_obj=mesh_path,
    alpha=0.0038,
    decimate_ratio=0.8
)
print(mesh_path)

 Chargement du nuage de points : c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\3d_reconstruction\point_cloud\iteration_35000\point_cloud_scaled.ply
Transformation dans l'espace de la caméra...
Génération du maillage Alpha Shape (alpha=0.0038m)...
Triangles générés : 16 456
 Décimation Quadrique... Réduction à 13 164 faces.
[Open3D WARNING] Write OBJ can not include triangle normals.
 Succès ! Maillage sauvegardé sous : c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\mesh.obj
c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\Plante_1_results\mesh.obj


## Visualisers

SIBR visualiser for gaussian splattings

In [29]:
import os
Viewer_path = (
            REPO_ROOT / "src/science_jubilee/Vision/3D_Reconstruction/Viewer/bin"
        )
        # Gaussian Viewer
os.system(
            f"cd {Viewer_path} && SIBR_gaussianViewer_app.exe -m {output_reconstruction }"
        )

1

Open3d visualiser for the created 3d mesh

In [26]:
import open3d as o3d
# Display the mesh
mesh = o3d.io.read_triangle_mesh(str(mesh_path))
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

# Features extracting

Thanks to themeshes and only using the point cloud of the reconstuction(center of the gaussians) we are able to extract information about our plant.

For ewample we have created a script to extract the leafs and find the horizontal ones

In [19]:
import itertools
import open3d as o3d

from science_jubilee.Vision.GS_Reconstruction.ingredients.extract_leafs import (
    run_extract_leaf_clusters,
)


mesh = o3d.io.read_triangle_mesh(str(mesh_path))
if not mesh.has_vertices():
    raise ValueError(f"Mesh has no vertices: {mesh_path}")

mesh_pcd = o3d.geometry.PointCloud(mesh.vertices)
parameter_grid = {
    "distance_threshold": [0.00001, 0.0001, 0.0005, 0.001, 0.003, 0.01, 0.03, 0.1],
    "min_points": [2, 3, 5, 10, 20, 50],
    "size_threshold": [0.00001, 0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.2],
    "shape_threshold": [0.20, 0.30, 0.40, 0.50, 0.65, 0.80, 0.95, 0.99],
    "height_ratio": [0.0, 0.1, 0.2, 0.4, 0.6, 0.8],
}
target_clusters = 15
search_results = []
total_combinations = 1
for values in parameter_grid.values():
    total_combinations *= len(values)

for combination_index, values in enumerate(
    itertools.product(*parameter_grid.values()), start=1
):
    parameters = dict(zip(parameter_grid, values))
    clusters = run_extract_leaf_clusters(pcd=mesh_pcd, **parameters)
    cluster_count = len(clusters)
    search_results.append(
        {
            "error": abs(cluster_count - target_clusters),
            "cluster_count": cluster_count,
            "parameters": parameters,
        }
    )
    if combination_index % 1000 == 0:
        print(f"Tested {combination_index}/{total_combinations} combinations")

search_results.sort(key=lambda result: (result["error"], -result["cluster_count"]))
best_result = search_results[0]
best_parameters = best_result["parameters"]
leaf_clusters = run_extract_leaf_clusters(pcd=mesh_pcd, **best_parameters)

print(f"Tested {total_combinations} combinations")
print(f"Best result: {best_result['cluster_count']} clusters")
print(f"Best parameters: {best_parameters}")
print("Top 10 candidates:")
for result in search_results[:10]:
    print(result["cluster_count"], result["parameters"])


Tested 1000/20736 combinations
Tested 2000/20736 combinations
Tested 3000/20736 combinations


KeyboardInterrupt: 

In [ ]:
import numpy as np
import open3d as o3d

from science_jubilee.Vision.GS_Reconstruction.ingredients.extract_leafs import (
    run_compute_leaf_normals,
    run_extract_normal_leafs,
    run_extract_leaf_clusters,
)


mesh = o3d.io.read_triangle_mesh(str(mesh_path))
if not mesh.has_vertices():
    raise ValueError(f"Mesh has no vertices: {mesh_path}")

mesh_pcd = o3d.geometry.PointCloud(mesh.vertices)
leaf_clusters = run_extract_leaf_clusters(
    pcd=mesh_pcd,
    distance_threshold=0.0092,
    min_points=20,
    size_threshold=1e-5,
    shape_threshold=0.98,
    height_ratio=0.1,
)

horizontal_threshold = 0.90
normals = run_compute_leaf_normals(leaf_clusters=leaf_clusters)
plant_z = np.array([0.0, 1.0, 0.0])
normal_z_dots = [
    np.nan if np.isscalar(normal) else abs(np.dot(normal, plant_z))
    for normal in normals
]
horizontal_leaf_clusters = run_extract_normal_leafs(
    leaf_clusters=leaf_clusters,
    horizontal_threshold=horizontal_threshold,
)
horizontal_leaf_ids = {id(leaf) for leaf in horizontal_leaf_clusters}

print("Normal dot Z for each leaf:")
for index, normal_z_dot in enumerate(normal_z_dots, start=1):
    print(f"Leaf {index}: {normal_z_dot:.4f}")
print(
    f"Horizontal leaves: {len(horizontal_leaf_clusters)}/{len(leaf_clusters)} "
    f"(|dot| >= {horizontal_threshold})"
)

geometries = [mesh]
#mesh.paint_uniform_color([0.65, 0.65, 0.65])

for index, leaf in enumerate(leaf_clusters, start=1):
    is_horizontal = id(leaf) in horizontal_leaf_ids
    leaf.paint_uniform_color([0.1, 0.8, 0.2] if is_horizontal else [0.7, 0.7, 0.7])
    bounding_box = leaf.get_axis_aligned_bounding_box()
    bounding_box.color = [1.0, 0.1, 0.1] if is_horizontal else [0.3, 0.3, 0.3]
    geometries.extend([leaf, bounding_box])

    normal = normals[index - 1]
    if np.isscalar(normal) or np.any(np.isnan(normal)):
        continue
    points = np.asarray(leaf.points)
    centroid = points.mean(axis=0)
    leaf_extent = np.linalg.norm(bounding_box.get_extent())
    arrow_length = max(leaf_extent * 0.35, 0.005)
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=arrow_length * 0.025,
        cone_radius=arrow_length * 0.08,
        cylinder_height=arrow_length * 0.75,
        cone_height=arrow_length * 0.25,
    )

    z_axis = np.array([0.0, 0.0, 1.0])
    rotation_vector = np.cross(z_axis, normal)
    rotation_vector_norm = np.linalg.norm(rotation_vector)
    rotation_dot = np.clip(np.dot(z_axis, normal), -1.0, 1.0)
    if rotation_vector_norm < 1e-12:
        rotation = np.eye(3) if rotation_dot >= 0 else np.diag([1.0, -1.0, -1.0])
    else:
        skew = np.array([
            [0.0, -rotation_vector[2], rotation_vector[1]],
            [rotation_vector[2], 0.0, -rotation_vector[0]],
            [-rotation_vector[1], rotation_vector[0], 0.0],
        ])
        rotation = (
            np.eye(3)
            + skew
            + skew @ skew * ((1.0 - rotation_dot) / rotation_vector_norm**2)
        )
    arrow.rotate(rotation, center=[0.0, 0.0, 0.0])
    arrow.translate(centroid)
    arrow.paint_uniform_color([1.0, 0.55, 0.0])
    geometries.append(arrow)

print("Orange arrows show the PCA normal of each leaf")
o3d.visualization.draw_geometries(
    geometries,
    window_name="Leaf PCA normals and bounding boxes",
    point_show_normal=False,
    mesh_show_back_face=True,
)